# Stage 3 — Supervised cross-lingual chemical NER on RuDReC

**Goal:** push RuDReC F1 from 0.290 (mBERT-DAPT zero-shot) towards **F1 ≈ 0.6**
in a controlled, lever-by-lever way.

**Why this is needed.** The pre-defence Stage 2 results showed two things at the
same time: DAPT gives a real, intrinsic-and-extrinsic-consistent gain on
multilingual encoders, and BioBERT cannot transfer to Russian at all because
its English WordPiece vocabulary cannot represent Cyrillic. To break out of
the 0.29 plateau we have to leave the strict zero-shot regime and use the
RuDReC train split — this is the natural Stage 3 of the project.

**Design rules (kept identical to Stage 1 and 2 where possible).**

- Single backbone of interest: `bert-base-multilingual-cased` (mBERT).
- BioBERT is kept only as a baseline control point — without vocab surgery it
  stays at F1 ≈ 0 on RuDReC and that fact is part of the thesis story.
- Reduce all NER tasks to the 3-tag BIO scheme `O / B-Chemical / I-Chemical`.
- One frozen fine-tuning recipe; differences in F1 are attributable to a
  single lever turned on or off, not to hyperparameter search.

**Five levers, additive, measured separately.**

1. **L0 — BioBERT control.** BioBERT-base zero-shot on RuDReC. Expected ≈ 0.
2. **L1 — mBERT supervised on RuDReC.** Train mBERT on the RuDReC train split
   from scratch (no DAPT). This is the honest supervised baseline.
3. **L2 — mBERT + DAPT on EN biomedical.** Add Stage-2 DAPT on BC5CDR +
   BC4CHEMD before the supervised step, to inject domain.
4. **L3 — mBERT + DAPT on EN biomedical + RuDReC raw text.** Extend DAPT to
   the raw Russian RuDReC text — language-side adaptation on top of L2.
5. **L4 — Class-weighted loss + slightly longer training.** Re-balance the
   over-represented `O` class and train for 5–8 epochs with early stopping.

The expected trajectory is roughly `0.29 (Stage 2) → 0.45 → 0.50 → 0.55 → 0.60+`.
The numbers below are predictions; the notebook fills in the actual values.

**Compute.** Designed for a free-tier Colab T4. Each lever is a separate cell
block so you can stop after any of them and still keep a clean comparison
table.


## 0 — Environment setup

> **Run this cell first, then click `Runtime → Restart session` once.**
> Colab ships pre-loaded `transformers` / `accelerate` versions that conflict
> with each other (`ImportError: cannot import name 'clear_device_cache'`).
> The pinned set below has been tested end-to-end on a free-tier T4.


In [ ]:
# 1) Clean install of a mutually compatible stack for Stage 3.
#    After this cell finishes -> Runtime -> Restart session, then run cell 2+.
!pip -q uninstall -y transformers accelerate datasets tokenizers huggingface_hub \
                     peft diffusers gradio
!pip -q install \
    "transformers==4.44.2" \
    "accelerate==0.34.2" \
    "datasets==2.21.0" \
    "tokenizers==0.19.1" \
    "huggingface_hub==0.24.7" \
    "seqeval==1.2.2" \
    "sentencepiece==0.2.0"
!pip -q install "peft==0.11.1"
print("\nDone. Now: Runtime -> Restart session, then continue from cell 2.")

In [ ]:
import os, json, math, random, time, gc
from pathlib import Path
from collections import Counter
from typing import List, Dict, Tuple

import numpy as np
import torch
from torch.utils.data import DataLoader

import transformers
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification, AutoModelForMaskedLM,
    DataCollatorForTokenClassification, DataCollatorForLanguageModeling,
    Trainer, TrainingArguments, get_linear_schedule_with_warmup,
)
from datasets import load_dataset, Dataset, DatasetDict
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

print("transformers", transformers.__version__)
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

# Repro
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# Frozen experimental protocol — identical for every lever.
CFG = dict(
    MAX_LEN     = 256,
    BATCH       = 16,
    LR_FT       = 3e-5,
    LR_MLM      = 5e-5,
    FT_EPOCHS   = 5,        # one knob compared to Stage 2 (was 3)
    MLM_EPOCHS  = 3,
    MLM_PROB    = 0.15,
    WARMUP      = 0.1,
    SEED        = SEED,
    BACKBONE    = "bert-base-multilingual-cased",
    BIOBERT     = "dmis-lab/biobert-base-cased-v1.1",
    LABELS      = ["O", "B-Chemical", "I-Chemical"],
)
LABEL2ID = {l: i for i, l in enumerate(CFG["LABELS"])}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
print(json.dumps(CFG, indent=2))


## 1 — Load datasets

**RuDReC** is the main target: Russian drug-related NER (chemical / drug entities).
We pull the raw JSONL release directly from the cimm-kzn GitHub mirror so the
notebook is self-contained. The original tags are mapped down to the 3-tag BIO
scheme used everywhere in the thesis.

**BC5CDR + BC4CHEMD** are reused for the EN biomedical DAPT corpus exactly as in
Stage 2 (parquet branches on the HF Hub).


In [ ]:
# RuDReC raw JSONL — same source as Stage 2 of the thesis.
import urllib.request, gzip, io

RUDREC_URL = "https://raw.githubusercontent.com/cimm-kzn/RuDReC/master/data/rudrec_annotated.json"
RUDREC_LOCAL = "/content/rudrec_annotated.json"

if not Path(RUDREC_LOCAL).exists():
    print("Downloading RuDReC ...")
    urllib.request.urlretrieve(RUDREC_URL, RUDREC_LOCAL)
print("RuDReC size, MB:", round(os.path.getsize(RUDREC_LOCAL) / 1e6, 2))


In [ ]:
import json
from datasets import Dataset

# RuDReC label set -> Chemical (BIO, 3 tags total: O / B-Chemical / I-Chemical)
DRUG_TYPES = {"Drugname", "Drugclass", "Drugform"}
LABEL2ID = {"O": 0, "B-Chemical": 1, "I-Chemical": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

def _char_spans_to_bio(text: str, entities: list) -> Tuple[List[str], List[str]]:
    # whitespace tokenization aligned to char offsets
    tokens, tok_spans = [], []
    i, n = 0, len(text)
    while i < n:
        if text[i].isspace():
            i += 1; continue
        j = i
        while j < n and not text[j].isspace():
            j += 1
        tokens.append(text[i:j]); tok_spans.append((i, j))
        i = j
    tags = ["O"] * len(tokens)
    for ent in entities:
        if ent.get("entity_type") not in DRUG_TYPES:
            continue
        s, e = int(ent["start"]), int(ent["end"])
        first = True
        for k, (a, b) in enumerate(tok_spans):
            if b <= s or a >= e:
                continue
            tags[k] = "B-Chemical" if first else "I-Chemical"
            first = False
    return tokens, tags

def load_rudrec(path: str) -> Dataset:
    sents = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            text = row.get("text", "")
            ents = row.get("entities", []) or []
            if not text:
                continue
            toks, tags = _char_spans_to_bio(text, ents)
            if toks:
                sents.append({"tokens": toks, "ner_tags": [LABEL2ID[t] for t in tags]})
    return Dataset.from_list(sents)

rudrec_full = load_rudrec(RUDREC_LOCAL)
print("RuDReC sentences:", len(rudrec_full))
from collections import Counter
print("Tag distribution (first 5000):",
      Counter(t for ex in rudrec_full.select(range(min(5000, len(rudrec_full))))
                for t in ex["ner_tags"]))

In [ ]:
# Train/val/test split — fixed seed, 70/10/20.
rudrec_split = rudrec_full.train_test_split(test_size=0.20, seed=SEED)
tmp = rudrec_split["train"].train_test_split(test_size=0.125, seed=SEED)  # 0.125*0.8 = 0.10
RUDREC = DatasetDict(
    train = tmp["train"],
    validation = tmp["test"],
    test = rudrec_split["test"],
)
print({k: len(v) for k, v in RUDREC.items()})


In [ ]:
# EN biomedical DAPT corpus — BC5CDR + BC4CHEMD raw text (same as Stage 2).
def _load_parquet(repo_id: str):
    return load_dataset(repo_id, split="train", revision="refs/convert/parquet")

try:
    bc5 = _load_parquet("tner/bc5cdr")
    bc4 = _load_parquet("disi-unibo-nlp/bc4chemd")
except Exception as e:
    print("Hub fetch failed:", e)
    bc5 = bc4 = None

def _to_text(ds):
    if ds is None: return []
    if "tokens" in ds.column_names:
        return [" ".join(ex["tokens"]) for ex in ds]
    if "text" in ds.column_names:
        return [ex["text"] for ex in ds]
    return [" ".join(map(str, ex.values())) for ex in ds]

en_dapt_text = _to_text(bc5) + _to_text(bc4)
print("EN DAPT sentences:", len(en_dapt_text))
print("Sample:", en_dapt_text[0][:160] if en_dapt_text else "(empty)")


## 2 — Tokenisation utilities (shared)

One BIO-aware tokeniser routine for every NER stage in the notebook. The
trick is the standard one: assign the label to the *first* sub-word of each
word, set `-100` on the other sub-word pieces and on `[CLS]` / `[SEP]`.


In [ ]:
def make_ner_tokeniser(tokenizer):
    def encode(batch):
        enc = tokenizer(
            batch["tokens"],
            is_split_into_words=True,
            truncation=True,
            max_length=CFG["MAX_LEN"],
        )
        all_labels = []
        for i, labels in enumerate(batch["ner_tags"]):
            word_ids = enc.word_ids(batch_index=i)
            prev = None
            new_labels = []
            for wid in word_ids:
                if wid is None:
                    new_labels.append(-100)
                elif wid != prev:
                    new_labels.append(labels[wid])
                else:
                    new_labels.append(-100)
                prev = wid
            all_labels.append(new_labels)
        enc["labels"] = all_labels
        return enc
    return encode

def compute_metrics_factory():
    def compute(p):
        preds, labels = p
        preds = np.argmax(preds, axis=-1)
        true_seqs, pred_seqs = [], []
        for pr, lb in zip(preds, labels):
            true_seqs.append([ID2LABEL[l] for l in lb if l != -100])
            pred_seqs.append([ID2LABEL[p_] for p_, l in zip(pr, lb) if l != -100])
        return dict(
            precision = precision_score(true_seqs, pred_seqs),
            recall    = recall_score(true_seqs, pred_seqs),
            f1        = f1_score(true_seqs, pred_seqs),
        )
    return compute


In [ ]:
RESULTS = {}  # collected at the end into a comparison table

def free_mem():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## L0 — BioBERT control point (expected F1 ≈ 0)

This is intentionally weak. The notebook keeps the result so the thesis can
say: *we tried BioBERT on Russian without vocabulary surgery; it stays at the
floor.* This is the same fact that motivates moving the cross-lingual work
onto a multilingual backbone.


In [ ]:
def evaluate_zero_shot(model_name: str, eval_dataset, label: str):
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForTokenClassification.from_pretrained(
        model_name, num_labels=len(CFG["LABELS"]),
        id2label=ID2LABEL, label2id=LABEL2ID,
    ).to(DEVICE).eval()

    enc = eval_dataset.map(make_ner_tokeniser(tok), batched=True,
                           remove_columns=eval_dataset.column_names)
    coll = DataCollatorForTokenClassification(tok)
    args = TrainingArguments(
        output_dir = f"/tmp/{label}",
        per_device_eval_batch_size = CFG["BATCH"],
        report_to = "none",
        fp16 = torch.cuda.is_available(),
    )
    trainer = Trainer(model=model, args=args, tokenizer=tok,
                      data_collator=coll, compute_metrics=compute_metrics_factory())
    m = trainer.evaluate(enc)
    print(label, "→", {k: round(v, 4) for k, v in m.items() if k.startswith("eval_")})
    free_mem()
    return m

m_bio = evaluate_zero_shot(CFG["BIOBERT"], RUDREC["test"], "L0_BioBERT_zero_shot")
RESULTS["L0_BioBERT_zero_shot"] = m_bio


## Helper — full supervised NER fine-tune on RuDReC

Used by levers L1, L2, L3, L4 with different starting weights and one extra
knob (class-weighted loss for L4).


In [ ]:
class WeightedNERModel(torch.nn.Module):
    """Wrapper that injects class weights into the token-classification loss."""
    def __init__(self, base, class_weights):
        super().__init__()
        self.base = base
        self.register_buffer("cw", torch.tensor(class_weights, dtype=torch.float32))

    def forward(self, **batch):
        labels = batch.pop("labels", None)
        out = self.base(**batch)
        logits = out.logits
        loss = None
        if labels is not None:
            loss = torch.nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)),
                labels.view(-1),
                weight=self.cw.to(logits.device),
                ignore_index=-100,
            )
        return {"loss": loss, "logits": logits}

    @property
    def config(self): return self.base.config
    def gradient_checkpointing_enable(self, **kw):
        return self.base.gradient_checkpointing_enable(**kw)
    def save_pretrained(self, *a, **kw):
        return self.base.save_pretrained(*a, **kw)


def estimate_class_weights(dataset, smoothing=0.1):
    counts = Counter()
    for ex in dataset:
        for t in ex["ner_tags"]:
            counts[t] += 1
    total = sum(counts.values())
    weights = []
    for i in range(len(CFG["LABELS"])):
        f = counts.get(i, 1) / total
        weights.append(1.0 / (f + smoothing))
    avg = sum(weights) / len(weights)
    return [w / avg for w in weights]


def supervised_fine_tune(model_or_name, label: str, *,
                         class_weighted: bool = False,
                         epochs: int = None):
    epochs = epochs or CFG["FT_EPOCHS"]
    tok_name = model_or_name if isinstance(model_or_name, str) else CFG["BACKBONE"]
    tok = AutoTokenizer.from_pretrained(tok_name, use_fast=True)

    if isinstance(model_or_name, str):
        model = AutoModelForTokenClassification.from_pretrained(
            model_or_name, num_labels=len(CFG["LABELS"]),
            id2label=ID2LABEL, label2id=LABEL2ID,
        )
    else:
        model = model_or_name

    if class_weighted:
        cw = estimate_class_weights(RUDREC["train"])
        print(f"[{label}] class weights:", [round(x, 3) for x in cw])
        model = WeightedNERModel(model, cw)

    enc = RUDREC.map(make_ner_tokeniser(tok), batched=True,
                     remove_columns=RUDREC["train"].column_names)
    coll = DataCollatorForTokenClassification(tok)

    args = TrainingArguments(
        output_dir=f"/content/{label}",
        num_train_epochs=epochs,
        per_device_train_batch_size=CFG["BATCH"],
        per_device_eval_batch_size=CFG["BATCH"],
        learning_rate=CFG["LR_FT"],
        warmup_ratio=CFG["WARMUP"],
        weight_decay=0.01,
        logging_steps=50,
        eval_strategy="epoch",
        save_strategy="no",
        report_to="none",
        seed=CFG["SEED"],
        fp16=torch.cuda.is_available(),
        remove_unused_columns=False,
    )

    trainer = Trainer(
        model=model, args=args, tokenizer=tok, data_collator=coll,
        train_dataset=enc["train"], eval_dataset=enc["validation"],
        compute_metrics=compute_metrics_factory(),
    )
    trainer.train()
    m_test = trainer.evaluate(enc["test"], metric_key_prefix="test")
    print(label, "→", {k: round(v, 4) for k, v in m_test.items() if k.startswith("test_")})
    RESULTS[label] = m_test
    free_mem()
    return trainer, m_test

## L1 — mBERT supervised on RuDReC (no DAPT)

Honest supervised baseline. This single lever is responsible for the largest
absolute jump in F1: going from zero-shot (~0.29 with Stage-2 DAPT) to
supervised typically adds +20–30 absolute F1 points, simply because the model
now sees a few thousand Russian-language token-level labels.


In [ ]:
_, m_L1 = supervised_fine_tune(CFG["BACKBONE"], "L1_mBERT_supervised")


## L2 — mBERT + DAPT (EN biomedical) → supervised RuDReC

Re-uses the Stage-2 DAPT recipe verbatim: 3 epochs of MLM on BC5CDR + BC4CHEMD
with `mlm_probability = 0.15`, `lr = 5e-5`. After DAPT we run the same
supervised fine-tune as L1.


In [ ]:
def run_dapt(text_corpus, base_model: str, label: str,
             epochs: int = None) -> str:
    """Domain-adaptive MLM. Returns the local path of the adapted checkpoint."""
    epochs = epochs or CFG["MLM_EPOCHS"]
    tok = AutoTokenizer.from_pretrained(base_model, use_fast=True)
    model = AutoModelForMaskedLM.from_pretrained(base_model)

    ds = Dataset.from_dict({"text": text_corpus})
    def tok_batch(b):
        return tok(b["text"], truncation=True, max_length=CFG["MAX_LEN"])
    ds = ds.map(tok_batch, batched=True, remove_columns=["text"])
    coll = DataCollatorForLanguageModeling(tok, mlm_probability=CFG["MLM_PROB"])

    out = f"/content/{label}"
    args = TrainingArguments(
        output_dir = out,
        per_device_train_batch_size = CFG["BATCH"],
        learning_rate = CFG["LR_MLM"],
        num_train_epochs = epochs,
        warmup_ratio = CFG["WARMUP"],
        save_strategy = "no",
        logging_strategy = "epoch",
        fp16 = torch.cuda.is_available(),
        report_to = "none",
        seed = CFG["SEED"],
    )
    Trainer(model=model, args=args, tokenizer=tok,
            data_collator=coll, train_dataset=ds).train()
    model.save_pretrained(out); tok.save_pretrained(out)
    free_mem()
    return out


In [ ]:
en_dapt_path = run_dapt(en_dapt_text, CFG["BACKBONE"], "mbert_dapt_en")
_, m_L2 = supervised_fine_tune(en_dapt_path, "L2_mBERT_DAPTen_supervised")


## L3 — mBERT + DAPT (EN biomed + RuDReC raw) → supervised RuDReC

The new lever introduced in Stage 3: *language-side* DAPT. We add the raw
RuDReC text — same examples that appear in supervised training, but only the
text without labels — to the DAPT corpus. Standard DAPT practice (Gururangan
et al. 2020); we document the transductive overlap in the thesis Limitations.

Expected: another +3–7 F1 over L2, mostly from better Russian-medical sub-word
distributions.


In [ ]:
ru_text = [" ".join(ex["tokens"]) for ex in RUDREC["train"]] + \
          [" ".join(ex["tokens"]) for ex in RUDREC["validation"]]
mixed_dapt_text = en_dapt_text + ru_text
print("Mixed DAPT corpus size:", len(mixed_dapt_text))

mix_dapt_path = run_dapt(mixed_dapt_text, CFG["BACKBONE"], "mbert_dapt_enru")
_, m_L3 = supervised_fine_tune(mix_dapt_path, "L3_mBERT_DAPTenru_supervised")


## L4 — L3 + class-weighted loss + 8 epochs (the F1 ≈ 0.6 attempt)

Final lever. Two things change versus L3:

- Cross-entropy is weighted by inverse token frequency (with a small smoothing
  constant) so that `B-Chemical` and `I-Chemical` are not drowned out by `O`.
- Train for 8 epochs instead of 5. RuDReC is small enough that the supervised
  signal converges late; we keep `eval_strategy = "epoch"` so the curve is
  visible.

This is the configuration the thesis reports as the headline supervised
cross-lingual result.


In [ ]:
_, m_L4 = supervised_fine_tune(
    mix_dapt_path,
    "L4_mBERT_DAPTenru_weighted_8ep",
    class_weighted=True,
    epochs=8,
)

## 5 — Comparison table and figure

The single source of truth for the thesis update. The Stage-2 number 0.290
(mBERT-DAPT zero-shot) is included for context.


In [ ]:
import pandas as pd

rows = [
    dict(setup="Stage 2 — mBERT + DAPT (EN), zero-shot RuDReC",
         f1=0.290, precision=None, recall=None, note="from thesis Stage 2"),
]
for k, m in RESULTS.items():
    p = m.get("test_precision", m.get("eval_precision"))
    r = m.get("test_recall",    m.get("eval_recall"))
    f = m.get("test_f1",        m.get("eval_f1"))
    rows.append(dict(setup=k, f1=f, precision=p, recall=r, note=""))

df = pd.DataFrame(rows)
df_show = df.copy()
for c in ["f1", "precision", "recall"]:
    df_show[c] = df_show[c].apply(lambda x: f"{x:.4f}" if isinstance(x, float) else "—")
df_show


In [ ]:
# Save the comparison table for inclusion in the thesis text and presentation.
df.to_csv("/content/stage3_results.csv", index=False)
print("Saved /content/stage3_results.csv")


In [ ]:
# Quick bar chart of supervised F1 across levers.
import matplotlib.pyplot as plt

bar_rows = df.dropna(subset=["f1"])
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(range(len(bar_rows)), bar_rows["f1"], color="#1E4D8C")
ax.set_xticks(range(len(bar_rows)))
ax.set_xticklabels([s.replace(" — ", "\n").replace(", ", "\n")
                    for s in bar_rows["setup"]],
                   rotation=0, fontsize=9)
ax.set_ylim(0, 1.0)
ax.set_ylabel("F1")
ax.set_title("Stage 3 — RuDReC F1 across levers")
for i, v in enumerate(bar_rows["f1"]):
    ax.text(i, v + 0.015, f"{v:.3f}", ha="center", fontsize=9)
ax.axhline(0.6, ls="--", color="#9C2A2A", lw=1)
ax.text(len(bar_rows) - 0.5, 0.61, "target 0.60", color="#9C2A2A", fontsize=9, ha="right")
plt.tight_layout(); plt.savefig("/content/stage3_results.png", dpi=140)
plt.show()


## 6 — Reading the result

If the trajectory looks like

```
L0  BioBERT zero-shot       ~ 0.00
L1  mBERT supervised        ~ 0.45 — 0.55
L2  + DAPT(EN)              + 0.02 — 0.04
L3  + DAPT(EN+RuDReC text)  + 0.03 — 0.06
L4  + class weights + 8 ep  + 0.02 — 0.04   →   F1 ≈ 0.55 — 0.65
```

then the thesis update is:

- **Headline.** Supervised mBERT with DAPT on a mixed EN+RU biomedical corpus
  and class-weighted loss reaches RuDReC F1 ≈ 0.6 — comparable to standard
  Russian biomedical NER baselines, on a free-tier T4.
- **What DAPT really does.** L1 → L3 isolates the DAPT contribution while the
  supervised signal is held constant. Any gain there is the cleanest evidence
  of DAPT we have.
- **What BioBERT cannot do.** L0 stays at the floor and is part of the story:
  English WordPiece is a hard structural ceiling for Russian biomedical NER.

If L1 already lands above 0.55, L2–L4 may each contribute less than predicted —
this is informative too, and means most of the F1 lives in the supervised
signal, not in DAPT. The thesis should report the table either way.
